Description
===========

This script generates the pyku configuration for the EUR-11 projection. This
script may need an update.

Reference
=========

* https://cordex.org/domains/

* https://euro-cordex.net/060378/index.php.en

* https://is-enes-data.github.io/cordex_archive_specifications.pdf

* https://github.com/WCRP-CORDEX/domain-tables/blob/main/CORDEX-CMIP6_domain_boundaries.csv

# Earth radius.

Without a clear reference, the CORDEX earth radius is taken from:

Reference: https://wcrp-cordex.github.io/archive-specifications/CORDEX-CMIP6_archiving_specifications_DD/#131-rotated-pole-coordinate-system

````
char crs ;
    crs:grid_mapping_name = "rotated_latitude_longitude" ;
    crs:grid_north_pole_latitude = 39.25 ;
    crs:grid_north_pole_longitude = -162. ;
    crs:earth_radius = 6371229. ;
````

## Converting grid_north_pole_longitude (CF-conform) to lon_0 (PROJ conform)

In [ ]:
grid_north_pole_longitude = -162.0

lon_0 = grid_north_pole_longitude + 180
if lon_0 > 180:
    lon_0 -= 360
print(lon_0)

## Projection definition

In [ ]:
def generate_pyku_configuration(grid):
    
    import cordex as cx
    import pyproj
    from pyresample import get_area_def

    proj_string = (
        "+proj=ob_tran +o_proj=longlat +R=6371229. +o_lat_p=39.25 +o_lon_p=0 +lon_0=18"
    )

    import cordex as cx

    domain_info = cx.domain_info(grid)
    domain_info

    # Set number of pixels
    # --------------------
    
    x_size = domain_info['nlon']
    y_size = domain_info['nlat']
    
    x_ll, y_ll = domain_info['ll_lon']-domain_info['dlon']/2., domain_info['ll_lat']-domain_info['dlat']/2.
    x_ur, y_ur = domain_info['ur_lon']+domain_info['dlon']/2., domain_info['ur_lat']+domain_info['dlat']/2.
        
    # There is a small error on the last decimale, likely due to float representation conversion
    x_ll, y_ll = round(x_ll, 10), round(y_ll, 10)
    x_ur, y_ur = round(x_ur, 10), round(y_ur, 10)
    
    # Create pyresample.AreaDefinition
    # --------------------------------
    
    area_id = domain_info['short_name']
    area_name = (
        f"{grid} rotated longitude latitude grid in accordance with CORDEX domains "
        "requirements https://cordex.org/domains/"
    )
    
    proj_id = None
    proj4_args = proj_string
    
    area_extent = (x_ll, y_ll, x_ur, y_ur)
    area_def = get_area_def(
        area_id,
        area_name,
        proj_id,
        proj4_args,
        x_size,
        y_size,
        area_extent
    )

    print(f"""\n>>>>>areas.yaml""")
    print(area_def.dump())

    print(f"""\n>>>>>areas_cf.yaml""")
    print(f"""
{grid}:
    crs_name: rotated_pole
    crs_data:
        grid_mapping_name: rotated_latitude_longitude
        grid_north_pole_latitude: 39.25
        grid_north_pole_longitude: -162.0
        earth_radius: 6371229.
        long_name: coordinates of the rotated North Pole
    y_coordinate: 'rlat'
    x_coordinate: 'rlon'
    CORDEX_domain: '{grid}'
    proj4: '{area_def.proj_str}'
    crs_wkt: '{area_def.crs_wkt}'
""")

In [ ]:
generate_pyku_configuration('EUR-11')
generate_pyku_configuration('EUR-44')